# Tutorial to run a magneto-rotational and detection simulation

In order to perform a magneto-rotational and detection simulation over a previously simulated dynamical database you can run the script:
```
python pypopsyn/simulator/simulate_population_magrot_det.py --dyn_data dyn_database --save_dir output/sim_magrot_det
```

To change the initial parameters, the user can directly modify the simulator configuration in `pypopsyn/simulator/config_simulator.py` or alternatively parsing a JSON file containing custom parameters for the simulation.

This script performs the following steps in a loop until the specified number of detections for each survey is reached (for example to match the number of detected pulsars in a given survey in the ATNF catalog):

1) It randomly samples batches of pulsars from the specified dynamical database.
   At the moment the batch size is set to 100000 and has been optimized for matching the radio observations with the Parkes telescope.
   This means that the dynamical database size should be bigger than this batch size, i.e., it should contain at least 100 times the neutron star number specified by the batch size.
   Simulating neutron stars in batches helps to speed up the simulation by evolving simultaneously an array of pulsars.

2) It selects pulsars that fall into the sky coverage of the surveys and are not further away than 35 kpc from the Sun and evolves their magnetic field, spin period and inclination angle.
   This pre-selection allows to not waste computational resources on pulsars that have no chance to be detected.

3) The emission geometry is modeled so that only pulsars emitting towards the Earth can be selected.
   A luminosity is associated with these pulsars and their flux is computed.

4) It finally applies the modeled surveys to select the pulsars that are detected according to the limiting flux of each survey.

This approach allows to find a posteriori the birth rate of the neutron stars by looking at the total number of neutron stars that have been created for a specified evolution time to reach the desired number of detections.

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg
from pypopsyn.simulator.simulate_population_magrot_det import simulate_population

## Setup and run the simulation

Change some parameters in the imported configuration file.

In [ ]:
cfg["B_initial_log10_mean"] = 13.1
cfg["B_initial_log10_sigma"] = 0.45
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38
cfg["a_late"] = -1.80

Specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_magrot_det"

Run the simulation.

In [ ]:
simulation_args = argparse.Namespace(
    dyn_data = "../../data/example_simulation_dyn",
    save_dir = output_dir,
    parameter_override = None
)
simulate_population(simulation_args)

## Read the simulation results

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

In [ ]:
print(
    f"Number of pulsars detected by PMPS: {len(data_PMPS)}"
)
print(
    f"Number of pulsars detected by SMPS: {len(data_SMPS)}"
)
print(
    f"Number of pulsars detected by HTRU low and mid latitude: {len(data_HTRU_low_mid)}"
)

config_filename = pathlib.Path().joinpath(output_dir, "configuration.json")
with config_filename.open("rt") as handle:
    config = json.load(handle, object_hook=collections.OrderedDict)

br_PMPS = config["birth_rate_PMPS_at_match"]
br_SMPS = config["birth_rate_SMPS_at_match"]
br_HTRU_low_mid = config["birth_rate_HTRU_low_mid_at_match"]

print(
    f"Pulsar birth rate predicted by PMPS: {br_PMPS}"
)
print(
    f"Pulsar birth rate predicted by SMPS: {br_SMPS}"
)
print(
    f"Pulsar birth rate predicted by HTRU low and mid latitude: {br_HTRU_low_mid}"
)


In [ ]:
# Extracting the parameters.
RA_pk_sim = data_PMPS["RA"]["[deg]"].to_numpy()
DEC_pk_sim = data_PMPS["DEC"]["[deg]"].to_numpy()
l_pk_sim = data_PMPS["l"]["[deg]"].to_numpy()
b_pk_sim = data_PMPS["b"]["[deg]"].to_numpy()
pmRA_pk_sim = data_PMPS["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_pk_sim = data_PMPS["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_pk_sim = data_PMPS["DM"]["[pc cm^-3]"].to_numpy()
dist_pk_sim = data_PMPS["d"]["[kpc]"].to_numpy()
P_pk_sim = data_PMPS["P"]["[s]"].to_numpy()
Pdot_pk_sim = data_PMPS["P_dot"]["[s s^-1]"].to_numpy()
S1400_pk_sim = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_pk_sim = data_PMPS["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_sw_sim = data_SMPS["RA"]["[deg]"].to_numpy()
DEC_sw_sim = data_SMPS["DEC"]["[deg]"].to_numpy()
l_sw_sim = data_SMPS["l"]["[deg]"].to_numpy()
b_sw_sim = data_SMPS["b"]["[deg]"].to_numpy()
pmRA_sw_sim = data_SMPS["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_sw_sim = data_SMPS["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_sw_sim = data_SMPS["DM"]["[pc cm^-3]"].to_numpy()
dist_sw_sim = data_SMPS["d"]["[kpc]"].to_numpy()
P_sw_sim = data_SMPS["P"]["[s]"].to_numpy()
Pdot_sw_sim = data_SMPS["P_dot"]["[s s^-1]"].to_numpy()
S1400_sw_sim = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_sw_sim = data_SMPS["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_htru_sim = data_HTRU_low_mid["RA"]["[deg]"].to_numpy()
DEC_htru_sim = data_HTRU_low_mid["DEC"]["[deg]"].to_numpy()
l_htru_sim = data_HTRU_low_mid["l"]["[deg]"].to_numpy()
b_htru_sim = data_HTRU_low_mid["b"]["[deg]"].to_numpy()
pmRA_htru_sim = data_HTRU_low_mid["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_htru_sim = data_HTRU_low_mid["pm_DEC"]["[mas yr^-1]"].to_numpy()
DM_htru_sim = data_HTRU_low_mid["DM"]["[pc cm^-3]"].to_numpy()
dist_htru_sim = data_HTRU_low_mid["d"]["[kpc]"].to_numpy()
P_htru_sim = data_HTRU_low_mid["P"]["[s]"].to_numpy()
Pdot_htru_sim = data_HTRU_low_mid["P_dot"]["[s s^-1]"].to_numpy()
S1400_htru_sim = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_htru_sim = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

## Plot the simulation results

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_sim,
    b_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    l_sw_sim,
    b_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    l_htru_sim,
    b_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.plot(0.0, 0.0, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim,
    Pdot_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim,
    Pdot_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.show()

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_pk_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
ax.hist(
    S1400_sw_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
ax.hist(
    S1400_htru_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:purple",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

To compare the simulation results with the real observations in the ATNF pulsar catalog you can use the notebook in `tutorials/analysis_notebook/survey_sim_vs_obs`.